In [36]:
# Vehicle Counting, Annotation and Training Pipeline
# -------------------------------------------------------------
# * 1️⃣  Frame‑level inference + vehicle counts per column
# * 2️⃣  Automatic extraction of YOLO‑format TXT labels (one per frame)
# * 3️⃣  Instant dataset split train / val / test and YAML file creation
# * 4️⃣  Kick‑off Ultralytics YOLOv8 training / fine‑tuning on your data
#
# Usage examples
# --------------
# Count + label only (no training):
#   python vehicle_counter_trainer.py --videos C:/Users/Janfl/Downloads/archive --output out --columns 3
#
# Count + label *and* fine‑tune a model for 50 epochs on the generated labels:
#   python vehicle_counter_trainer.py --videos C:/Users/Janfl/Downloads/archive \
#       --output out --columns 3 \
#       --train --epochs 50 --batch 16 --img 640
#
# All heavy‑lifting happens in the main() at the bottom.
# -------------------------------------------------------------
from types import SimpleNamespace 
import argparse
import sys
import csv
import shutil
from pathlib import Path
from random import shuffle

import cv2
import numpy as np
from ultralytics import YOLO

# -------------- CONFIG DEFAULTS --------------------------------------------
VEHICLE_CATEGORIES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
ID_MAP = { coco_id: idx for idx, coco_id in enumerate(VEHICLE_CATEGORIES) }
DEFAULT_MODEL = "yolov8n.pt"

In [37]:
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser("Vehicle counting, annotation & training pipeline")
    # inference / labelling
    p.add_argument("--videos", type=Path, required=True, help="Folder with input videos")
    p.add_argument("--output", type=Path, default=Path("output"), help="Root output directory")
    p.add_argument("--model", type=str, default=DEFAULT_MODEL, help="YOLO checkpoint to start from")
    p.add_argument("--conf", type=float, default=0.25, help="Confidence threshold")
    p.add_argument("--stride", type=int, default=5, help="Process every n‑th frame")
    p.add_argument("--columns", type=int, default=3, help="# vertical columns (lanes)")
    p.add_argument("--save-video", action="store_true", help="Save annotated mp4 next to labels")
    # training
    p.add_argument("--train", action="store_true", help="Finetune YOLO on generated labels")
    p.add_argument("--epochs", type=int, default=1)
    p.add_argument("--batch", type=int, default=8)
    p.add_argument("--img", type=int, default=640, help="Training image size")
    p.add_argument("--device", type=str, default="cuda", help="cuda / cpu / 0,1 …")
    return p.parse_args()

In [38]:
# -------------------- HELPERS ---------------------------------------------

def make_dirs(base: Path, video_name: str):
    framelabel_dir = base / "labels" / video_name
    frameimg_dir = base / "images" / video_name
    framelabel_dir.mkdir(parents=True, exist_ok=True)
    frameimg_dir.mkdir(parents=True, exist_ok=True)
    return frameimg_dir, framelabel_dir


def column_index(x_center: float, frame_w: int, n_cols: int) -> int:
    return int((x_center / frame_w) * n_cols)


In [ ]:
# -------------------- INFERENCE & LABELLING -------------------------------
def process_videos(args):
    args.output.mkdir(parents=True, exist_ok=True)
    model = YOLO(args.model)
    summary_rows = []

    for vid_path in sorted(args.videos.glob("*")):
        if vid_path.suffix.lower() not in {".mp4", ".avi", ".mov", ".mkv"}:
            continue

        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            print(f"[WARN] cannot open {vid_path}")
            continue

        vname = vid_path.stem
        img_dir, label_dir = make_dirs(args.output, vname)

        # one counter list per class and lane
        counts    = {n: [0] * args.columns for n in VEHICLE_CATEGORIES.values()}
        # one set per lane to remember which BoT‑SORT IDs we’ve already counted
        seen_ids  = [set() for _ in range(args.columns)]

        frame_idx = 0
        writer    = None

        # --------------- video writer (optional) -----------------
        if args.save_video:
            fps    = cap.get(cv2.CAP_PROP_FPS)
            width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            out_p  = args.output / f"{vname}_annotated.mp4"
            writer = cv2.VideoWriter(str(out_p), fourcc, fps, (width, height))
        # ---------------------------------------------------------

        colors = {"car": (0,255,0), "motorcycle": (0,255,255),
                  "bus": (255,0,0), "truck": (255,0,255)}

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_idx % args.stride == 0:                     # process only every n‑th frame
                h, w = frame.shape[:2]

                #-------- detection + BoT‑SORT tracking ----------
                = model.track(
                    frame,
                    imgsz   = args.img,
                    conf    = args.conf,
                    tracker ='botsort.yaml',
                    persist = True,
                    verbose = False)[0]

                list = res.boxes.id         # ← grab the list once (may be None)

                _lines = []

                i, (box, cid, conf) in enumerate(
                    zip(res.boxes.xyxy.cpu().numpy(),
                        res.boxes.cls.cpu().numpy().astype(int),
                        res.boxes.conf.cpu().numpy())):

                if cid not in VEHICLE_CATEGORIES:
                    continue

                x1, y1, x2, y2 = box
                xc, yc   = (x1 + x2) / 2, (y1 + y2) / 2
                lane_idx = column_index(xc, w, args.columns)
                track_id = int(tid_list[i]) if tid_list is not None else -1
                cname    = VEHICLE_CATEGORIES[cid]

                # -------- UNIQUE‑COUNT LOGIC ----------
                if track_id != -1:                          # only if tracker produced an ID
                    if track_id not in seen_ids[lane_idx]:
                        seen_ids[lane_idx].add(track_id)
                        counts[cname][lane_idx] += 1
                # --------------------------------------

                # write YOLO txt
                new_id = ID_MAP[cid]
                yolo_lines.append(
                    f"{new_id} {xc/w:.6f} {yc/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")

                # draw bbox with label
                col = colors[cname]
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
                cv2.putText(frame, f"{cname} {conf:.2f}", (int(x1), int(y1) - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

                # draw counts on frame
                y_off = 50
                for cls, lanes in counts.items():
                    for l, cnt in enumerate(lanes):
                        cv2.putText(frame, f"{cls} L{l}: {cnt}", (10,y_off),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                        y_off += 25

                # save frame + label
                fname = f"{frame_idx:06}.jpg"
                cv2.imwrite(str(img_dir / fname), frame)
                with open(label_dir / f"{frame_idx:06}.txt", "w") as f:
                    f.write("\n".join(yolo_lines))

            # always write frame if saving video
            if writer is not None:
                writer.write(frame)

            frame_idx += 1
            if frame_idx % 200 == 0:
                print(f"[{vname}] processed {frame_idx} raw frames")

        # --- end‑of‑video housekeeping ---
        cap.release()
        if writer:
            writer.release()
            print(f"Saved annotated video → {out_p}")

        # one summary row per video
        row = {"video": vname}
        for cls, lanes in counts.items():
            for l, cnt in enumerate(lanes):
                row[f"{cls}_lane{l}"] = cnt
            row[f"{cls}_total"] = sum(lanes)
        summary_rows.append(row)
        print(f"Finished {vname} counts → {counts}")

    # write global CSV
    if summary_rows:
        csv_path = args.output / "summary.csv"
        with open(csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
            w.writeheader(); w.writerows(summary_rows)
        print(f"Saved summary CSV → {csv_path}")


In [40]:
# -------------------- DATASET PREP FOR TRAINING ---------------------------

def split_dataset(root: Path, splits=(0.8, 0.1, 0.1)):
    """
    Split only the original per-video folders under images/, then
    mirror that structure under images/train, val, test (and same for labels/).
    """
    img_root = root / "images"
    lbl_root = root / "labels"

    # 1) find only your original video folders
    video_dirs = [
        d for d in img_root.iterdir()
        if d.is_dir() and d.name not in ("train", "val", "test")
    ]

    # 2) collect all frames from them
    all_imgs = []
    for vd in video_dirs:
        all_imgs += list(vd.glob("*.jpg"))

    shuffle(all_imgs)
    n = len(all_imgs)
    n_train = int(n * splits[0])
    n_val = int(n * splits[1])

    splits_and_files = [
      ("train", all_imgs[:n_train]),
      ("val", all_imgs[n_train:n_train+n_val]),
      ("test", all_imgs[n_train+n_val:])
    ]

    # 3) make split dirs under images/ and labels/
    for split_name, _ in splits_and_files:
        (img_root / split_name).mkdir(parents=True, exist_ok=True)
        (lbl_root / split_name).mkdir(parents=True, exist_ok=True)

    # 4) move each image + its corresponding .txt label
    for split_name, img_list in splits_and_files:
        for img_path in img_list:
            video_folder = img_path.parent.name
            img_name = img_path.name
            label_path = lbl_root / video_folder / img_path.with_suffix(".txt").name

            if not label_path.exists():
                print(f"Warning: Missing label for {img_path}")
                continue

            # target sub‐dirs
            dst_img_dir = img_root / split_name / video_folder
            dst_lbl_dir = lbl_root / split_name / video_folder
            dst_img_dir.mkdir(exist_ok=True, parents=True)
            dst_lbl_dir.mkdir(exist_ok=True, parents=True)

            # move
            shutil.move(str(img_path), str(dst_img_dir / img_name))
            shutil.move(str(label_path), str(dst_lbl_dir / img_name.replace(".jpg", ".txt")))

    # 5) report
    counts = {split: len(imgs) for split, imgs in splits_and_files}
    print("Dataset split done:", counts)


def create_yaml(root: Path):
    yaml_path = root / "dataset.yaml"
    content = f"""path: {root.resolve()}  # root dir of dataset
train: images/train
val: images/val
test: images/test
nc: {len(VEHICLE_CATEGORIES)}
names: [{', '.join(VEHICLE_CATEGORIES.values())}]
"""
    yaml_path.write_text(content)
    return yaml_path


In [41]:
# -------------------- TRAIN ------------------------------------------------

def train_yolo(yaml_file: Path, args):
    print("Starting YOLO training...")
    model = YOLO(args.model)
    model.train(
        data=str(yaml_file),
        epochs=args.epochs,
        batch=args.batch,
        imgsz=args.img,
        device=args.device,
        name="veh_finetune",
        exist_ok=True,  # Avoid conflicts with existing runs
        patience=10     # Early stopping if no improvement
    )
    print(f"Training completed! Results are in {Path('runs/detect/veh_finetune')}")
    
    # Optional: Validate the trained model
    print("Validating trained model...")
    metrics = model.val(data=str(yaml_file))
    print(f"Validation metrics: {metrics}")
    
    return Path('runs/detect/veh_finetune/weights/best.pt')

In [42]:
def main(args=None):
    """
    If args is None we are in CLI mode -> parse command‑line.
    Otherwise we received a pre‑built args namespace (e.g. from a notebook).
    """
    if args is None:
        args = parse_args()   # <-- regular argparse path

    print("Starting vehicle counting pipeline with settings:")
    print(f"- Processing videos from: {args.videos}")
    print(f"- Output to:            {args.output}")
    print(f"- Video stride:         {args.stride} frames")
    print(f"- Lane columns:         {args.columns}")
    print(f"- Model:                {args.model}\n")

    process_videos(args)

    if args.train:
        print(f"\nPreparing dataset for training with {args.epochs} epochs …")
        split_dataset(args.output)
        yaml_file = create_yaml(args.output)
        best_weights = train_yolo(yaml_file, args)
        print(f"Best weights saved to: {best_weights}")


# --------- Notebook‑friendly argument block -----------
JupyterArgs = SimpleNamespace(
    videos     = Path(r"C:\Users\Janfl\Downloads\archive"),      # <= your AVI(s)
    output     = Path(r"C:\Users\Janfl\Downloads\archive_out"),
    model      = "yolov8n.pt",
    conf       = 0.25,
    stride     = 1,          # every 2‑nd frame
    columns    = 3,
    save_video = True,
    train      = False,      # True → fine‑tune after labelling
    epochs     = 1,
    batch      = 16,
    img        = 640,
    device     = 0           # GPU id or "cpu"
)

# ---------- call main() -------------
main(JupyterArgs)            # run inside notebook
# main()                      # run from command line (will parse sys.argv)

Starting vehicle counting pipeline with settings:
- Processing videos from: C:\Users\Janfl\Downloads\archive
- Output to:            C:\Users\Janfl\Downloads\archive_out
- Video stride:         1 frames
- Lane columns:         3
- Model:                yolov8n.pt

[39031] processed 200 raw frames
[39031] processed 400 raw frames
[39031] processed 600 raw frames
[39031] processed 800 raw frames
[39031] processed 1000 raw frames


TypeError: 'NoneType' object is not subscriptable